# Master Notebook for Edge Classification (Branch Congestion)
This master notebook trains and compares multiple Graph Neural Network architectures (GCN, GraphSAGE, GAT, MPNN, ECGNN) to predict congestion across the branches of the IEEE 30-bus grid.

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "pandas>=2.2.3", "pandapower", "numpy", "torch", "torchvision",
    "torchaudio", "torch_geometric", "h5py", "optuna", "matplotlib"])

0

In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split
import pandapower.networks as nw

c:\Users\i34005\OneDrive - Wood Mackenzie Limited\CSML\csml-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. PyTorch Geometric Dataset

In [3]:
from ieee_dataset import IEEECongestionCSVDataset

csv_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "congestion_dataset_v3.csv")
dataset = IEEECongestionCSVDataset(csv_path)

train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dataset: {len(dataset)} samples  (train={train_size}, val={val_size}, test={test_size})")
print(f"Using device: {device}")

Dataset: 50000 samples  (train=35000, val=7500, test=7500)
Using device: cpu


2. Train Multiple Models

Iterating through multiple model types

In [ ]:
from models.gcn_model import GCNEdgePredictor
from models.graphsage_model import GraphSAGEEdgePredictor
from models.gat_model import GATEdgePredictor
from models.mpnn_model import MPNNEdgePredictor
from models.ecgnn_model import ECGNNEdgePredictor

MODEL_TYPES = ['GCN']

model_classes = {
    'GCN': GCNEdgePredictor,
    'GraphSAGE': GraphSAGEEdgePredictor,
    'GAT': GATEdgePredictor,
    'MPNN': MPNNEdgePredictor,
    'ECGNN': ECGNNEdgePredictor
}

epochs = 3
all_histories = {}
all_test_metrics = {}

for m_type in MODEL_TYPES:
    print(f"\n{'='*40}\nTraining {m_type}\n{'='*40}")
    ModelClass = model_classes[m_type]
    
    model = ModelClass(
        in_channels=5,
        hidden_channels=64,
        branch_u=dataset.branch_u,
        branch_v=dataset.branch_v,
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    pos_weight = torch.tensor([10.0]).to(device)
    criterion = nn.BCEWithLogitsLoss(reduction='none', pos_weight=pos_weight)
    
    history = {"epoch": [], "train_loss": [], "val_acc": []}
    
    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        total_loss, n_batches = 0.0, 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            y = batch.y.view(-1, 1)
            mask = batch.status.view(-1, 1)
            loss = (criterion(out, y) * mask).sum() / (mask.sum() + 1e-8)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        avg_loss = total_loss / n_batches
        
        # --- Validate ---
        model.eval()
        correct, total_active = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch)
                y = batch.y.view(-1, 1)
                mask = batch.status.view(-1, 1)
                preds = (out > 0).float()
                correct += ((preds == y).float() * mask).sum().item()
                total_active += mask.sum().item()
        val_acc = correct / total_active
        
        history["epoch"].append(epoch)
        history["train_loss"].append(avg_loss)
        history["val_acc"].append(val_acc)
        
        if epoch % 5 == 0 or epoch == epochs:
            print(f"Epoch {epoch:3d}/{epochs} | loss={avg_loss:.4f} | val_acc={val_acc:.4f}")
            
    all_histories[m_type] = history
    
    # --- Test ---
    model.eval()
    correct, total_active = 0, 0
    tp, fp, fn = 0, 0, 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out = model(batch)
            y = batch.y.view(-1, 1)
            mask = batch.status.view(-1, 1)
            preds = (out > 0).float()
            masked_preds = preds * mask
            masked_y = y * mask
            correct += ((preds == y).float() * mask).sum().item()
            total_active += mask.sum().item()
            tp += ((masked_preds == 1) & (masked_y == 1)).float().sum().item()
            fp += ((masked_preds == 1) & (masked_y == 0)).float().sum().item()
            fn += ((masked_preds == 0) & (masked_y == 1)).float().sum().item()
            
    test_acc = correct / total_active if total_active > 0 else 0
    precision = tp / (tp + fp + 1e-8) if tp + fp > 0 else 0
    recall = tp / (tp + fn + 1e-8) if tp + fn > 0 else 0
    f1 = 2 * precision * recall / (precision + recall + 1e-8) if precision + recall > 0 else 0
    
    all_test_metrics[m_type] = {
        "acc": test_acc, "precision": precision, "recall": recall, "f1": f1
    }
    print(f"Test Accuracy: {test_acc:.4f} | F1: {f1:.4f}")


Training GCN
Epoch   5/30 | loss=0.2878 | val_acc=0.9135


3. Compare Models

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for m_type, hist in all_histories.items():
    ax1.plot(hist["epoch"], hist["train_loss"], label=m_type)
    ax2.plot(hist["epoch"], hist["val_acc"], label=m_type)

ax1.set(xlabel="Epoch", ylabel="Loss", title="Training Loss")
ax1.legend()
ax2.set(xlabel="Epoch", ylabel="Accuracy", title="Validation Accuracy")
ax2.legend()
fig.tight_layout()
plt.show()

print("\n--- Test Set Results ---")
metrics_df = pd.DataFrame(all_test_metrics).T
metrics_df = metrics_df[['acc', 'f1', 'precision', 'recall']]
display(metrics_df.sort_values(by='f1', ascending=False))

### 4. Hyperparameter Tuning (Optuna) for Trained Models
Optuna searches over hidden_channels, batch_size, learning rate, and dropout — training on the **train** split and evaluating on the **validation** split for all models listed in `MODEL_TYPES`.

In [ ]:
import optuna
import pandas as pd

best_hyperparams = {}

for TUNE_MODEL in MODEL_TYPES:
    print(f"\n{'='*50}")
    print(f"Tuning hyperparameters for {TUNE_MODEL}...")
    print(f"{'='*50}")
    
    OptunaModelClass = model_classes[TUNE_MODEL]
    
    def objective(trial):
        hidden_channels = trial.suggest_categorical("hidden_channels", [32, 64, 128])
        batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.4)
        
        tune_train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        tune_val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        trial_model = OptunaModelClass(
            in_channels=5,
            hidden_channels=hidden_channels,
            branch_u=dataset.branch_u,
            branch_v=dataset.branch_v,
            dropout_rate=dropout_rate,
        ).to(device)
        
        opt = torch.optim.Adam(trial_model.parameters(), lr=lr)
        pos_w = torch.tensor([10.0]).to(device)
        crit = nn.BCEWithLogitsLoss(reduction='none', pos_weight=pos_w)
        
        for epoch in range(12):
            trial_model.train()
            for batch in tune_train_loader:
                batch = batch.to(device)
                opt.zero_grad()
                out = trial_model(batch)
                y = batch.y.view(-1, 1)
                mask = batch.status.view(-1, 1)
                loss = (crit(out, y) * mask).sum() / (mask.sum() + 1e-8)
                loss.backward()
                opt.step()
                
        trial_model.eval()
        correct, total_active = 0, 0
        with torch.no_grad():
            for batch in tune_val_loader:
                batch = batch.to(device)
                out = trial_model(batch)
                y = batch.y.view(-1, 1)
                mask = batch.status.view(-1, 1)
                preds = (out > 0).float()
                correct += ((preds == y).float() * mask).sum().item()
                total_active += mask.sum().item()
        return correct / total_active
        
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=5)  # Set n_trials higher for a real search
    print(f"\nBest val accuracy for {TUNE_MODEL}: {study.best_value:.4f}")
    print(f"Best params for {TUNE_MODEL}: {study.best_params}")
    
    best_hyperparams[TUNE_MODEL] = {
        'best_val_acc': study.best_value,
        'params': study.best_params
    }

print("\n\n--- OPTUNA RESULTS SUMMARY ---")
for model_name, info in best_hyperparams.items():
    print(f"{model_name} (Acc: {info['best_val_acc']:.4f})  Params: {info['params']}")